In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.neighbors import KernelDensity
import re
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
import itertools
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from scipy.stats import ks_2samp



In [ ]:
outpath = "./test_data/step2_compare_ic_bw_region_v3"
os.makedirs(outpath,exist_ok=True)
#read input
inpath = "./deal_bindingdb_alldata.csv"
data = pd.read_csv(inpath,header=0)

In [18]:
data.head()

,Ligand SMILES,UniProt (SwissProt) Recommended Name of Target Chain,UniProt (SwissProt) Primary ID of Target Chain,dim1,dim2,region,density,log_density,IC50_dealed2
0,CC(O)(CS(=O)(=O)c1ccc(F)cc1)C(=O)Nc1ccc(C#N)c(...,Androgen receptor,P10275,-2.716189,4.470571,R1,0.033852,-3.385762,54.000000
1,N#Cc1ccc(cc1)C(c1ccc(cc1)C#N)n1cncn1,Aromatase,P11511,-0.039476,0.565852,R1,0.007116,-4.945343,0.700000
2,COc1ccc(cc1F)-c1c(nc(N2CCC(N)CC2)n(C)c1=O)-c1c...,Lysine-specific histone demethylase 1A,O60341,-2.816713,3.484625,R1,0.047409,-3.048936,0.300000
3,CC(=O)N1CCc2c(C1)[nH]nc2C(=O)N1CCC(CC1)c1ccc(F...,Retinol-binding protein 4,P02753,-0.873268,3.343346,R1,0.027553,-3.591641,2.824087
4,COc1ccccc1CN[C@H]1C2CCN(CC2)[C@H]1C(c1ccccc1)c...,Substance-P receptor,P25103,-1.235433,1.633217,R1,0.026360,-3.635899,0.200000


In [19]:
def extract_data(uni_id,data):
    #get target names in data
    if uni_id  in data["UniProt (SwissProt) Primary ID of Target Chain"].to_list():
        mydata1 = data[data["UniProt (SwissProt) Primary ID of Target Chain"]==uni_id].copy()
        init_name = mydata1["UniProt (SwissProt) Recommended Name of Target Chain"].values[0].replace(" ","_")
        # name format
        tar_name = re.sub(r'[^0-9a-zA-Z]+', '_', init_name)
        print(f"target 1 is {tar_name}")
        print(f"data long is {mydata1.shape[0]}")
        return tar_name,mydata1
    else:
        print("no matching data")
        return None,None

In [20]:

targets_dict = {
   "P42345" :"MTOR","Q02750":"MEK1"

 }

In [21]:
targets =[ i for i in targets_dict.keys()]


In [22]:
target_names = []
extracted_data = []
for target in targets:
    print(f"extracting for {target}")
    target_name, target_data =  extract_data(target,data)
    target_names.append(target_name)
    extracted_data.append(target_data)

extracting for P42345
target 1 is Serine_threonine_protein_kinase_mTOR
data long is 3499
extracting for Q02750
target 1 is Dual_specificity_mitogen_activated_protein_kinase_kinase_1
data long is 923


In [23]:
sub_data = pd.concat(extracted_data)

In [24]:
sub_data["simple_name"] =sub_data["UniProt (SwissProt) Primary ID of Target Chain"].apply(
    lambda x:targets_dict[x]
)

In [25]:
def extrac_ic(density_data,top_ratio):
    """compare IC50 in the top-density region"""

    order = ["R3", "R1", "R2"] #R3-> R1 -> R2

    #for region, df_sub in subset1.groupby("region"):
    top_density = []
    for region in order:
        df_sub = density_data[density_data["region"] == region]
        density = df_sub["density"].values
    
      
        #df_with_density = df_sub.copy()
    
        # ----------select top density----------
        threshold = np.quantile(density, 1 - top_ratio)
        core_mask = density >= threshold
        df_core = df_sub.loc[core_mask]
        df_core.columns = df_sub.columns.to_list()
        top_density.append(df_core)
    saved_df = pd.concat(top_density)
    return saved_df

In [26]:
def bootstrap_median_ci(values, n_boot=5000, ci=95):
    values = np.array(values)
    boot_medians = []
    for _ in range(n_boot):
        sample = np.random.choice(values, size=len(values), replace=True)
        boot_medians.append(np.median(sample))
    lower = np.percentile(boot_medians, (100-ci)/2)
    upper = np.percentile(boot_medians, 100-(100-ci)/2)
    return np.median(values), lower, upper



In [27]:
def bootstrap_mean_ci(values, n_boot=5000, ci=95):
    boot_means = [np.mean(np.random.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    lower = np.percentile(boot_means, (100-ci)/2)
    upper = np.percentile(boot_means, 100-(100-ci)/2)
    return np.mean(values), lower, upper


In [28]:

# get target list
target_names_list = list(targets_dict.values())  # ['MTOR', 'MEK1']

# set top_ratio
top_ratio = 0.45

# 
for target_name in target_names_list:
    print(f"\n{'='*60}")
    print(f"Processing target: {target_name}")
    print(f"{'='*60}")
    
    # extract data
    target_data = sub_data[sub_data["simple_name"] == target_name]
    
    #get top-density
    plot_data = extrac_ic(target_data, top_ratio)
    
    
    outpath3 = os.path.join(outpath, target_name)
    os.makedirs(outpath3, exist_ok=True)
    
    # median and CI
    results_med = []
    for region, group in plot_data.groupby("region"):
        vals = group["IC50_dealed2"].dropna()
        median, ci_low, ci_high = bootstrap_median_ci(vals)
        print(f"{region}: median={median:.3f}, 95% CI=({ci_low:.3f}, {ci_high:.3f})")
        results_med.append({
            "region": region,
            "median": median,
            "CI_lower": ci_low,
            "CI_upper": ci_high,
            "n": len(vals)
        })
    
    # save
    med_data = pd.DataFrame(results_med)
    med_data.to_csv(f"{outpath3}/{target_name}_{top_ratio}_median_data.csv", index=False)
    
    # mean and CI
    results_mean = []
    for region, group in plot_data.groupby("region"):
        vals = group["IC50_dealed2"].dropna()
        mean, ci_low, ci_high = bootstrap_mean_ci(vals)
        print(f"{region}: mean={mean:.3f}, 95% CI=({ci_low:.3f}, {ci_high:.3f})")
        results_mean.append({
            "region": region,
            "mean": mean,
            "CI_lower": ci_low,
            "CI_upper": ci_high,
            "n": len(vals)
        })
    
    # save
    mean_data = pd.DataFrame(results_mean)
    mean_data.to_csv(f"{outpath3}/{target_name}_{top_ratio}_mean_data.csv", index=False)
    
    #
    plot_data.to_csv(f"{outpath3}/{target_name}_{str(top_ratio)}_density_plotdata.csv")
    
    # KS test and FDR
    regions = plot_data["region"].unique()
    values = {
        r: plot_data.loc[plot_data["region"] == r, "IC50_dealed2"].dropna()
        for r in regions
    }
    
    pairs = list(itertools.combinations(regions, 2))
    raw_p = []
    results = []
    
    for a, b in pairs:
        stat, p = ks_2samp(values[a], values[b])
        raw_p.append(p)
        
        med_a = np.median(values[a])
        med_b = np.median(values[b])
        direction = f"{a} > {b}" if med_a > med_b else f"{a} < {b}"
        
        results.append({
            "comparison": f"{a} vs {b}",
            "KS_stat": stat,
            "raw_p": p,
            "median_a": med_a,
            "median_b": med_b,
            "direction": direction
        })
    
    # FDR
    rej, p_adj, _, _ = multipletests(raw_p, method="fdr_bh")
    
    for i, r in enumerate(results):
        r["FDR_p"] = p_adj[i]
        r["significant"] = rej[i]
    
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{outpath3}/{target_name}_{top_ratio}_pvalue_data_kw_FDR.csv")
    
    print(f"\nAfter FDR adjustment for {target_name}:")
    for (a, b), p0, p1, sig in zip(pairs, raw_p, p_adj, rej):
        med_a = np.median(values[a])
        med_b = np.median(values[b])
        direction = f"{a} > {b}" if med_a > med_b else f"{a} < {b}"
        print(f"{a} vs {b}: raw p={p0:.3e}, FDR p={p1:.3e}, significant={sig}, median: {med_a:.3f} vs {med_b:.3f}, direction: {direction}")
    
    # evaluate hotspot
    med_data_dict = {}
    for region in regions:
        region_data = plot_data[plot_data["region"] == region]["IC50_dealed2"].dropna()
        if len(region_data) > 0:
            med_data_dict[region] = {
                "median": np.median(region_data),
                "mean": np.mean(region_data)
            }
    
    # sort by median 
    sorted_by_median = sorted(med_data_dict.items(), key=lambda x: x[1]["median"])
    
    # 
    min_median = sorted_by_median[0][1]["median"]
    candidates = []
    
    
    for region, stats in sorted_by_median:
        if stats["median"] == min_median:
            candidates.append((region, stats))
        else:
            break
    
    # 
    if len(candidates) == 1:
        opt = candidates[0][0]
    else:
        #if all medians are equal, sort by mean
        sorted_by_mean = sorted(candidates, key=lambda x: x[1]["mean"])
        opt = sorted_by_mean[0][0]
    
    print(f"\nOptimal region for {target_name}: {opt} (median={med_data_dict[opt]['median']:.3f}, mean={med_data_dict[opt]['mean']:.3f})")
    
    # save anchor molecules in hotspot 
    outpath4 = os.path.join(outpath3, "for_reinforce")
    os.makedirs(outpath4, exist_ok=True)
    
    save_data =target_data[target_data["region"] == opt]
    save_data.to_csv(f"{outpath4}/{target_name}_{str(top_ratio)}_{opt}_density_plotdata.csv")
    
    #save all statistical data
    region_stats = []
    for region in regions:
        if region in med_data_dict:
            region_stats.append({
                "target": target_name,
                "region": region,
                "median": med_data_dict[region]["median"],
                "mean": med_data_dict[region]["mean"],
                "is_optimal": region == opt
            })
    
    region_stats_df = pd.DataFrame(region_stats)
    region_stats_df.to_csv(f"{outpath3}/{target_name}_region_stats.csv", index=False)
    
    print(f"Finished processing {target_name}")


Processing target: MTOR
R1: median=20.000, 95% CI=(17.080, 24.000)
R2: median=25.050, 95% CI=(21.000, 35.000)
R3: median=46.000, 95% CI=(30.000, 65.500)
R1: mean=85.297, 95% CI=(74.724, 96.363)
R2: mean=142.644, 95% CI=(116.418, 170.923)
R3: mean=131.917, 95% CI=(109.959, 155.637)

After FDR adjustment for MTOR:
R3 vs R1: raw p=1.115e-06, FDR p=3.344e-06, significant=True, median: 46.000 vs 20.000, direction: R3 > R1
R3 vs R2: raw p=3.712e-02, FDR p=3.712e-02, significant=True, median: 46.000 vs 25.050, direction: R3 > R2
R1 vs R2: raw p=1.522e-02, FDR p=2.283e-02, significant=True, median: 20.000 vs 25.050, direction: R1 < R2

Optimal region for MTOR: R1 (median=20.000, mean=85.297)
Finished processing MTOR

Processing target: MEK1
R1: median=57.500, 95% CI=(43.000, 73.000)
R2: median=49.000, 95% CI=(39.000, 57.000)
R3: median=25.000, 95% CI=(12.000, 250.000)
R1: mean=134.996, 95% CI=(109.081, 162.624)
R2: mean=156.935, 95% CI=(121.675, 196.089)
R3: mean=233.829, 95% CI=(146.165, 330